In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler,OrdinalEncoder


In [4]:
df = pd.read_csv("../DATASET/Dataset.csv")
df.head()

,Unnamed: 0,Id,year,brand,full_model_name,model_name,price,distance_travelled(kms),fuel_type,city,...,new and less used,inv_car_price,inv_car_dist,inv_car_age,inv_brand,std_invprice,std_invdistance_travelled,std_invrank,best_buy1,best_buy2
0,0,0,2016,Honda,Honda Brio S MT,Brio,425000.0,9680.0,Petrol,Mumbai,...,0,2.352941e-06,0.000103,0.200000,0.142857,0.143417,0.035730,0.132143,677.134239,0.000000
1,1,1,2012,Nissan,Nissan Sunny XV Diesel,Sunny,325000.0,119120.0,Diesel,Mumbai,...,0,3.076923e-06,0.000008,0.111111,0.090909,0.188859,0.002496,0.079545,37.501318,0.000000
2,2,2,2017,Toyota,Toyota Fortuner 2.8 4x2 MT [2016-2020],Fortuner,2650000.0,64593.0,Diesel,Thane,...,0,3.773585e-07,0.000015,0.250000,1.000000,0.019416,0.004978,1.000000,96.646020,0.000000
3,3,3,2017,Mercedes-Benz,Mercedes-Benz E-Class E 220d Expression [2019-...,E-Class,4195000.0,25000.0,Diesel,Mumbai,...,1,2.383790e-07,0.000040,0.250000,0.500000,0.010692,0.013563,0.493750,71.604306,71.604306
4,4,4,2012,Hyundai,Hyundai Verna Fluidic 1.6 CRDi SX,Verna,475000.0,23800.0,Diesel,Mumbai,...,0,2.105263e-06,0.000042,0.111111,0.071429,0.127871,0.014269,0.059821,109.150857,0.000000


In [5]:
extra_column=[
    'Unnamed : 0',
    'Id',
    'inv_car_price',
    'inv_car_dist',
    'inv_car_age', 
    'inv_brand',
    'std_invprice',
    'std_invdistance_travelled',
    'std_invrank', 
    'best_buy1',
    'best_buy2',
    'car_age'
    
]
df = df.drop(df.columns[0], axis=1)
df = df.drop(columns=extra_column, errors="ignore")
df.head()

,year,brand,full_model_name,model_name,price,distance_travelled(kms),fuel_type,city,brand_rank,distance below 30k km,new and less used
0,2016,Honda,Honda Brio S MT,Brio,425000.0,9680.0,Petrol,Mumbai,7,1,0
1,2012,Nissan,Nissan Sunny XV Diesel,Sunny,325000.0,119120.0,Diesel,Mumbai,11,0,0
2,2017,Toyota,Toyota Fortuner 2.8 4x2 MT [2016-2020],Fortuner,2650000.0,64593.0,Diesel,Thane,1,0,0
3,2017,Mercedes-Benz,Mercedes-Benz E-Class E 220d Expression [2019-...,E-Class,4195000.0,25000.0,Diesel,Mumbai,2,1,1
4,2012,Hyundai,Hyundai Verna Fluidic 1.6 CRDi SX,Verna,475000.0,23800.0,Diesel,Mumbai,14,1,0


In [6]:
df.columns

Index(['year', 'brand', 'full_model_name', 'model_name', 'price',
       'distance_travelled(kms)', 'fuel_type', 'city', 'brand_rank',
       'distance below 30k km', 'new and less used'],
      dtype='object')

In [7]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df.select_dtypes(include=["object"]).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [8]:
# Feature 1 created
current_year = 2026
df["car_age"] = current_year - df['year']


In [9]:
# Feature 2 created
df["km_per_year"] = df["distance_travelled(kms)"] / (
    df["car_age"] + 1
)


In [10]:
# Feature 3 created
df["usage_tier"] = pd.cut(
    df["distance_travelled(kms)"],
    bins=[-1, 30000, 75000, np.inf],
    labels=["low_mileage", "moderate_mileage", "high_mileage"],
)
df.head()

,year,brand,full_model_name,model_name,price,distance_travelled(kms),fuel_type,city,brand_rank,distance below 30k km,new and less used,car_age,km_per_year,usage_tier
0,2016,Honda,Honda Brio S MT,Brio,425000.0,9680.0,Petrol,Mumbai,7,1,0,10,880.000000,low_mileage
1,2012,Nissan,Nissan Sunny XV Diesel,Sunny,325000.0,119120.0,Diesel,Mumbai,11,0,0,14,7941.333333,high_mileage
2,2017,Toyota,Toyota Fortuner 2.8 4x2 MT [2016-2020],Fortuner,2650000.0,64593.0,Diesel,Thane,1,0,0,9,6459.300000,moderate_mileage
3,2017,Mercedes-Benz,Mercedes-Benz E-Class E 220d Expression [2019-...,E-Class,4195000.0,25000.0,Diesel,Mumbai,2,1,1,9,2500.000000,low_mileage
4,2012,Hyundai,Hyundai Verna Fluidic 1.6 CRDi SX,Verna,475000.0,23800.0,Diesel,Mumbai,14,1,0,14,1586.666667,low_mileage


In [11]:
def cap_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    df[column] = df[column].clip(lower=lower_bound, upper=upper_bound)
    return df
    
df = cap_outliers(df, 'distance_travelled(kms)')


Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

df['price'] = df['price'].clip(lower=lower_limit, upper=upper_limit)


max_rank_limit = df['brand_rank'].quantile(0.99)
df['brand_rank'] = df['brand_rank'].clip(upper=max_rank_limit)


df['new and less used'] = df['new and less used'].astype(int)


Q1_age = df['car_age'].quantile(0.25)
Q3_age = df['car_age'].quantile(0.75)
IQR_age = Q3_age - Q1_age

upper_age = Q3_age + 1.5 * IQR_age
lower_age = max(0, Q1_age - 1.5 * IQR_age)  # Age negative nahi ho sakti

df['car_age'] = df['car_age'].clip(lower=lower_age, upper=upper_age)


Q1_km = df['km_per_year'].quantile(0.25)
Q3_km = df['km_per_year'].quantile(0.75)
IQR_km = Q3_km - Q1_km

upper_km = Q3_km + 1.5 * IQR_km
lower_km = max(0, Q1_km - 1.5 * IQR_km)  # Age negative nahi ho sakti

df['km_per_year'] = df['km_per_year'].clip(lower=lower_km, upper=upper_km)

# Verification: Check min/max after capping
print("Cap hone ke baad ranges:")
print(f"car_age -> Min: {df['car_age'].min()}, Max: {df['car_age'].max()}")
print(f"km_per_year -> Min: {df['km_per_year'].min()}, Max: {df['km_per_year'].max():.2f}")

Cap hone ke baad ranges:
car_age -> Min: 5.0, Max: 20.5
km_per_year -> Min: 58.333333333333336, Max: 10000.00


In [12]:
# num_cols = ['cgpa', 'backlogs', 'internships', 'certifications', 
#             'coding_skills', 'communication_skills', 'aptitude_score', 'projects']
# num_cols = df.select_dtypes(include=["int64", "float64"]).columns
# cat_cols = df.select_dtypes(include=["object"]).columns

# # 1. ALL OUTLIERS IN ONE VIEW
# plt.figure(figsize=(8, 10))
# for i, col in enumerate(num_cols, 1):
#     plt.subplot(4, 3, i)
#     sns.boxplot(y=df[col], color='lightcoral')
#     plt.title(f'Outliers: {col}')

# plt.tight_layout()
# plt.show()

In [13]:
cat_cols = df.select_dtypes(include=["object"]).columns
print(cat_cols)

Index(['brand', 'full_model_name', 'model_name', 'fuel_type', 'city'], dtype='object')


In [14]:
df.columns

Index(['year', 'brand', 'full_model_name', 'model_name', 'price',
       'distance_travelled(kms)', 'fuel_type', 'city', 'brand_rank',
       'distance below 30k km', 'new and less used', 'car_age', 'km_per_year',
       'usage_tier'],
      dtype='object')

In [15]:
y = df['price']

drop_cols = [
    'price',                   
    'year',                  
    'full_model_name',         
    'distance below 30k km',   
    'usage_tier'              
]

X = df.drop(columns=[col for col in drop_cols if col in df.columns])

print("Target (y) shape:", y.shape)
print("Features (X) shape:", X.shape)
print("\nFinal X Columns:")
print(X.columns.tolist())


Target (y) shape: (1725,)
Features (X) shape: (1725, 9)

Final X Columns:
['brand', 'model_name', 'distance_travelled(kms)', 'fuel_type', 'city', 'brand_rank', 'new and less used', 'car_age', 'km_per_year']


In [16]:
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)
print(X_train.shape)
X_test.shape

(1380, 9)


(345, 9)

In [17]:


# scaler = StandardScaler()
# # df['distance_travelled(kms)'] = scaler.fit_transform(df['distance_travelled(kms)'])
# df['car_age'] = scaler.fit_transform(df['car_age'])
# df['km_per_year'] = scaler.fit_transform(df['km_per_year'])
# df.head()

categorical_cols = ['fuel_type']
numerical_cols = ['distance_travelled(kms)','car_age', 'km_per_year','brand_rank']
text_cols = ["brand", "model_name", "city"]
binary_cols = ["new and less used"]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols),
        ("text",OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),text_cols),
        ("pass","passthrough",binary_cols),
    ]
)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

joblib.dump(preprocessor, '../models/scaler_pipeline.pkl')
print("Preprocessor saved successfully as '../models/scaler_pipeline.pkl!")



Preprocessor saved successfully as '../models/scaler_pipeline.pkl!


In [18]:
model = RandomForestRegressor()

model.fit(X_train_processed,y_train)

y_pred = model.predict(X_test_processed)
print("Mean Absolute Error:",mean_absolute_error(y_pred,y_test))
print("Mean Squared Error:",np.sqrt(mean_squared_error(y_pred,y_test)))
print("R2 Score:",r2_score(y_pred,y_test))
      

Mean Absolute Error: 200308.57648953304
Mean Squared Error: 363883.5916008542
R2 Score: 0.8735969016929657


In [19]:
# impoertant Colummn or features

# Extract target feature names from ColumnTransformer
feature_names = preprocessor.get_feature_names_out()

# Match feature names with importance values
importances = model.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("\n--- Top 5 Features Driving Car Price ---")
print(feature_importance_df.head(5).to_string(index=False))


--- Top 5 Features Driving Car Price ---
                     Feature  Importance
                 text__brand    0.292401
             num__brand_rank    0.238554
                num__car_age    0.196394
            text__model_name    0.094859
num__distance_travelled(kms)    0.058374


In [20]:
# 1. Define a synthetic car sample
new_car = pd.DataFrame([{
    'distance_travelled(kms)': 45000,
    'car_age': 4,
    'km_per_year': 11250,
    'brand_rank': 2,
    'fuel_type': 'Petrol',
    'brand': 'Toyota',
    'model_name': 'Corolla',
    'city': 'Ahmedabad',
    'new and less used': 0
}])

# 2. Preprocess raw data using saved/fitted ColumnTransformer
new_car_processed = preprocessor.transform(new_car)

# 3. Predict price
predicted_price = model.predict(new_car_processed)[0]

print(f"\nPredicted Price for New Car: ₹{predicted_price:,.2f}")


Predicted Price for New Car: ₹2,309,100.00


In [21]:
joblib.dump(model, '../models/car_price_model.pkl')

print("Model saved successfully!")

Model saved successfully!


In [22]:

loaded_preprocessor = joblib.load('../models/scaler_pipeline.pkl')
loaded_model = joblib.load('../models/car_price_model.pkl')

sample_car = pd.DataFrame([{
    'distance_travelled(kms)': 35000,
    'car_age': 3,
    'km_per_year': 11666,
    'brand_rank': 1,
    'fuel_type': 'Petrol',
    'brand': 'Hyundai',
    'model_name': 'i20',
    'city': 'Mumbai',
    'new and less used': 0
}])

# Transform raw input using the loaded preprocessor
sample_processed = loaded_preprocessor.transform(sample_car)

# Generate price prediction
predicted_price = loaded_model.predict(sample_processed)[0]

print(f"Predicted Market Price: ₹{predicted_price:,.2f}")

Predicted Market Price: ₹2,881,740.00


In [23]:
df["brand"].unique()

array(['Honda', 'Nissan', 'Toyota', 'Mercedes-Benz', 'Hyundai',
       'Maruti Suzuki', 'Renault', 'Volkswagen', 'Skoda', 'BMW', 'Tata',
       'Audi', 'Bentley', 'Ford', 'Mahindra', 'Jaguar', 'Lamborghini',
       'MINI', 'Land Rover', 'Chevrolet', 'Datsun', 'Jeep', 'Porsche',
       'Volvo', 'MG', 'Lexus', 'Mitsubishi', 'Kia', 'Fiat', 'Isuzu',
       'Mahindra-Renault'], dtype=object)

In [26]:
df["city"].unique()

array(['Mumbai', 'Thane', 'Dehradun', 'Navi Mumbai', 'Delhi', 'Noida',
       'Ghaziabad', 'Panchkula', 'Faridabad', 'Agra', 'Lucknow',
       'Bangalore', 'Hyderabad', 'Chennai', 'Pune'], dtype=object)

In [ ]:
city